## Parsing methods Experimentation

Quantitative Comparison Metrics:
- Accuracy in table and graphical extraction
- Ability to handle complex formatting (Images, tables, equations)
- Preservation of document structure
- Speed of parsing

Evaluation techniques:
- Manual visual inspection
- End-to-End Testing: Run the entire RAG pipeline with different parsing strategies and evaluating the final output.

Approach in choosing parsers:
1. Use external benchmarks (OmniDocBench, READOC) to pick top 2-3 parsers
2. Run a small local benchmark (5-10 representative papers).
3. Measure our own criteria (Quantitative Comparison Metrics)
4. Pick the best parser strategy for our use case

Useful resources: 
- https://www.reddit.com/r/LangChain/comments/1ef12q6/the_rag_engineers_guide_to_document_parsing/
- https://github.com/opendatalab/OmniDocBench

---

### Quick Start
1. Run cells 2 to 6 in order.
2. Start MLflow UI in terminal:
   - `mlflow ui --backend-store-uri /Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns`
3. Review parser artifacts in MLflow UI:
   - Experiment -> Individual runs -> Artifacts
4. Log manual inspection scores: eg
   - `log_manual_inspection_by_parser("pymupdf4llm", 4, "Good structure, minor table loss", run_ids)`

In [1]:
import pandas as pd
from datetime import datetime

# Benchmark metadata for representative complex-structure papers
# Keep this as the source of truth for which papers are included in the benchmark set.
benchmark_rows = [
    {
        "paper_id": "P1",
        "title": "Relaxation-Informed Training of Neural Network Surrogate Models",
        "file_name": "Relaxation-Informed Training of Neural Network Surrogate Models.pdf",
        "arxiv_url": "https://arxiv.org/abs/2604.22746v1",
        "year": 2026,
        "domain": "cs.LG",
        "complexity_type": "equations|figures|optimization",
        "notes": "Math-heavy, good stress test for equation preservation.",
    },
    {
        "paper_id": "P2",
        "title": "Long-Tail Internet Photo Reconstruction",
        "file_name": "Long-Tail Internet Photo Reconstruction.pdf",
        "arxiv_url": "https://arxiv.org/abs/2604.22714v1",
        "year": 2026,
        "domain": "cs.CV",
        "complexity_type": "figures|images|captions",
        "notes": "Figure-heavy CV layout.",
    },
    {
        "paper_id": "P3",
        "title": "Generative Modeling of Neurodegenerative Brain Anatomy with 4D Longitudinal Diffusion Model",
        "file_name": "Generative Modeling of Neurodegenerative Brain Anatomy with 4D Longitudinal Diffusion Model.pdf",
        "arxiv_url": "https://arxiv.org/abs/2604.22700v1",
        "year": 2026,
        "domain": "cs.CV",
        "complexity_type": "figures|tables|medical-imaging",
        "notes": "Medical AI paper with dense visual content.",
    },
    {
        "paper_id": "P4",
        "title": "Contexts are Never Long Enough: Structured Reasoning for Scalable Question Answering over Long Document Sets",
        "file_name": "Contexts are Never Long Enough- Structured Reasoning for Scalable Question Answering over Long Document Sets.pdf",
        "arxiv_url": "https://arxiv.org/abs/2604.22294v1",
        "year": 2026,
        "domain": "cs.CL",
        "complexity_type": "long-doc|tables|structured-reasoning",
        "notes": "Long context document with likely complex table structure.",
    },
    {
        "paper_id": "P5",
        "title": "STEM: Structure-Tracing Evidence Mining for Knowledge Graphs-Driven Retrieval-Augmented Generation",
        "file_name": "STEM- Structure-Tracing Evidence Mining for Knowledge Graphs-Driven Retrieval-Augmented Generation.pdf",
        "arxiv_url": "https://arxiv.org/abs/2604.22282v1",
        "year": 2026,
        "domain": "cs.CL",
        "complexity_type": "figures|graphs|multi-hop-reasoning",
        "notes": "Graph/diagram heavy RAG paper.",
    },
]

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df["added_at"] = datetime.utcnow().isoformat()
benchmark_df

,paper_id,title,file_name,arxiv_url,year,domain,complexity_type,notes,added_at
0,P1,Relaxation-Informed Training of Neural Network...,Relaxation-Informed Training of Neural Network...,https://arxiv.org/abs/2604.22746v1,2026,cs.LG,equations|figures|optimization,"Math-heavy, good stress test for equation pres...",2026-06-04T05:37:16.962514
1,P2,Long-Tail Internet Photo Reconstruction,Long-Tail Internet Photo Reconstruction.pdf,https://arxiv.org/abs/2604.22714v1,2026,cs.CV,figures|images|captions,Figure-heavy CV layout.,2026-06-04T05:37:16.962514
2,P3,Generative Modeling of Neurodegenerative Brain...,Generative Modeling of Neurodegenerative Brain...,https://arxiv.org/abs/2604.22700v1,2026,cs.CV,figures|tables|medical-imaging,Medical AI paper with dense visual content.,2026-06-04T05:37:16.962514
3,P4,Contexts are Never Long Enough: Structured Rea...,Contexts are Never Long Enough- Structured Rea...,https://arxiv.org/abs/2604.22294v1,2026,cs.CL,long-doc|tables|structured-reasoning,Long context document with likely complex tabl...,2026-06-04T05:37:16.962514
4,P5,STEM: Structure-Tracing Evidence Mining for Kn...,STEM- Structure-Tracing Evidence Mining for Kn...,https://arxiv.org/abs/2604.22282v1,2026,cs.CL,figures|graphs|multi-hop-reasoning,Graph/diagram heavy RAG paper.,2026-06-04T05:37:16.962514


In [2]:
import glob
import mlflow
import time
from pathlib import Path

#pdfs = glob.glob("../data/raw/*pdf")
#target_filepath = glob.glob("../data/raw/attention-is-all-you-need.pdf")[0]
#target_filepath

target_filepath = "../data/raw/attention-is-all-you-need-full.pdf"
target_file= Path(target_filepath).name
target_file

/Users/carlychinsekyi/Downloads/GitHub/papermind/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'attention-is-all-you-need-full.pdf'

In [3]:
import mlflow
import time
from pathlib import Path

mlflow.set_tracking_uri("file:///Users/carlychinsekyi/Downloads/GitHub/papermind/notebooks/mlruns")
# TODO: use sqlite instead of filestore -> mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Papermind_Parsing_Audit")


def run_parsing_experiment(parser_name, pdf_path, parse_func, parser_version, paper_id=None):
    """
    Standardized wrapper to run a parser and log results to MLflow.
    Returns the MLflow run_id so manual scoring can be done without copy-paste.
    """
    pdf_path = str(pdf_path)
    stem = Path(pdf_path).stem
    run_name = f"{paper_id}_{parser_name}_{stem}" if paper_id else f"{parser_name}_{stem}"

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        # 1. Log Metadata
        mlflow.log_param("parser", parser_name)
        mlflow.log_param("parser_version", parser_version)
        mlflow.log_param("file_name", Path(pdf_path).name)
        mlflow.log_param("pdf_path", pdf_path)
        if paper_id:
            mlflow.log_param("paper_id", paper_id)

        # 2. Execute & Time
        start_time = time.time()
        try:
            markdown_text = parse_func(pdf_path)
            duration = time.time() - start_time

            # 3. Log Performance Metrics
            mlflow.log_metric("latency_sec", round(duration, 2))
            mlflow.log_metric("char_count", len(markdown_text))

            # 4. Save Artifact (The actual Markdown)
            output_file = f"experiments/outputs/{stem}_{parser_name}_result.md"
            Path(output_file).parent.mkdir(parents=True, exist_ok=True)
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(markdown_text)
            mlflow.log_artifact(output_file)

            print(f"{parser_name} complete for {Path(pdf_path).name}. Open {output_file} to inspect.")
            print(f"Run ID: {run_id}")

        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_text(str(e), "error_log.txt")
            print(f"{parser_name} failed: {e}")

        return run_id


def log_manual_inspection(run_id, score, comments):
    """
    Call this after you've looked at the output file.
    score: 1-5 (1=Trash, 5=Perfect)
    """
    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric("manual_visual_score", score)
        mlflow.set_tag("manual_inspection_notes", comments)
        print(f"Logged score {score} for run {run_id}")


def log_manual_inspection_by_parser(parser_name, score, comments, run_ids_dict, paper_id=None):
    """
    Convenience helper: log manual score using parser name.
    Supports both:
    - flat dict: run_ids[parser_name] = run_id
    - nested dict: run_ids[paper_id][parser_name] = run_id
    """
    if paper_id is not None:
        paper_runs = run_ids_dict.get(paper_id, {})
        run_id = paper_runs.get(parser_name)
        if not run_id:
            raise ValueError(f"No run_id found for parser '{parser_name}' under paper_id '{paper_id}'.")
    else:
        run_id = run_ids_dict.get(parser_name)
        if not run_id:
            raise ValueError(f"No run_id found for parser '{parser_name}'.")

    log_manual_inspection(run_id, score, comments)

/Users/carlychinsekyi/Downloads/GitHub/papermind/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [4]:
import subprocess
from pathlib import Path

import pymupdf4llm
from docling.document_converter import DocumentConverter


def parse_with_pymupdf4llm(pdf_path: str) -> str:
    return pymupdf4llm.to_markdown(pdf_path)


def parse_with_docling(pdf_path: str) -> str:
    converter = DocumentConverter()
    doc = converter.convert(pdf_path).document
    return doc.export_to_markdown()


def parse_with_mineru(pdf_path: str) -> str:
    # Use per-paper output directories to avoid stale markdown collisions.
    output_dir = Path("experiments/outputs/mineru") / Path(pdf_path).stem
    output_dir.mkdir(parents=True, exist_ok=True)

    # hybrid-auto-engine mixes OCR/VLM and text-based parsing paths automatically.
    cmd = [
        "mineru",
        "-p", str(pdf_path),
        "-o", str(output_dir),
        "--backend", "hybrid-auto-engine",
        "-f", "True",
        "-t", "True",
    ]
    subprocess.run(cmd, check=True)

    md_files = sorted(output_dir.rglob("*.md"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not md_files:
        raise FileNotFoundError(f"No markdown output found in {output_dir}")

    return md_files[0].read_text(encoding="utf-8")


PARSER_CONFIG = {
   # "pymupdf4llm": (parse_with_pymupdf4llm, "1.27.2.2"),
 #   "docling": (parse_with_docling, "2.91.0"),
    "mineru": (parse_with_mineru, "3.1.5")
}

In [5]:
raw_dir = Path("../data/raw")
run_ids = {}
run_records = []

top2_papers = 2
i = 0
for row in benchmark_df.itertuples(index=False):
    # analyse 2 papers only for now
    i += 1
    if i > top2_papers: break
    
    pdf_path = raw_dir / row.file_name
    if not pdf_path.exists():
        print(f"Skipping {row.paper_id}: file not found -> {pdf_path}")
        continue

    run_ids[row.paper_id] = {}
    for parser_name, (parse_func, parser_version) in PARSER_CONFIG.items():
        run_id = run_parsing_experiment(
            parser_name,
            str(pdf_path),
            parse_func,
            parser_version,
            paper_id=row.paper_id,
        )
        run_ids[row.paper_id][parser_name] = run_id
        run_records.append(
            {
                "paper_id": row.paper_id,
                "file_name": row.file_name,
                "parser": parser_name,
                "parser_version": parser_version,
                "run_id": run_id,
            }
        )

run_results_df = pd.DataFrame(run_records)
print("Run IDs nested by paper:", run_ids)
run_results_df

2026-06-04 13:37:21.798 | INFO     | mineru.cli.client:run_orchestrated_cli:905 - Started local mineru-api at http://127.0.0.1:58208
2026-06-04 13:37:22.877 | INFO     | __main__:create_app:234 - Request concurrency limited to 1
INFO:     Started server process [91765]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:58208 (Press CTRL+C to quit)


Start MinerU FastAPI Service: http://127.0.0.1:58208
API documentation: http://127.0.0.1:58208/docs


2026-06-04 13:37:23.823 | INFO     | mineru.cli.client:run_planned_task:785 - Submitting batch 1/1 | 1 document, 35 pages in this batch | 35 pages total | task#1 [Relaxation-Informed Training of Neural Network Surrogate Models]
2026-06-04 13:37:23.882 | INFO     | mineru.utils.engine_utils:get_vlm_engine:34 - Using transformers as the inference engine for VLM.
2026-06-04 13:37:23.883 | ERROR    | __main__:_process_task:1150 - Async task failed: 3c37b770-9c72-4269-9774-fba7abae2687
Traceback (most recent call last):

  File "/Users/carlychinsekyi/Downloads/GitHub/papermind/.venv/lib/python3.11/site-packages/mineru/cli/common.py", line 74, in _load_hybrid_analyze_entrypoint
    hybrid_analyze = importlib.import_module("mineru.backend.hybrid.hybrid_analyze")
                     │         └ <function import_module at 0x10286af20>
                     └ <module 'importlib' from '/Users/carlychinsekyi/.local/share/uv/python/cpython-3.11.8-macos-aarch64-none/lib/python3.11/impor...
  File "/

mineru failed: Command '['mineru', '-p', '../data/raw/Relaxation-Informed Training of Neural Network Surrogate Models.pdf', '-o', 'experiments/outputs/mineru/Relaxation-Informed Training of Neural Network Surrogate Models', '--backend', 'hybrid-auto-engine', '-f', 'True', '-t', 'True']' returned non-zero exit status 1.


2026-06-04 13:37:26.988 | INFO     | mineru.cli.client:run_orchestrated_cli:905 - Started local mineru-api at http://127.0.0.1:58216
2026-06-04 13:37:28.038 | INFO     | __main__:create_app:234 - Request concurrency limited to 1
INFO:     Started server process [91780]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:58216 (Press CTRL+C to quit)


Start MinerU FastAPI Service: http://127.0.0.1:58216
API documentation: http://127.0.0.1:58216/docs


2026-06-04 13:37:29.017 | INFO     | mineru.cli.client:run_planned_task:785 - Submitting batch 1/1 | 1 document, 16 pages in this batch | 16 pages total | task#1 [Long-Tail Internet Photo Reconstruction]
2026-06-04 13:37:29.112 | INFO     | mineru.utils.engine_utils:get_vlm_engine:34 - Using transformers as the inference engine for VLM.
2026-06-04 13:37:29.113 | ERROR    | __main__:_process_task:1150 - Async task failed: 854dd38e-ce5e-4744-a8d9-4672382a450b
Traceback (most recent call last):

  File "/Users/carlychinsekyi/Downloads/GitHub/papermind/.venv/lib/python3.11/site-packages/mineru/cli/common.py", line 74, in _load_hybrid_analyze_entrypoint
    hybrid_analyze = importlib.import_module("mineru.backend.hybrid.hybrid_analyze")
                     │         └ <function import_module at 0x10040ef20>
                     └ <module 'importlib' from '/Users/carlychinsekyi/.local/share/uv/python/cpython-3.11.8-macos-aarch64-none/lib/python3.11/impor...
  File "/Users/carlychinsekyi/.lo

mineru failed: Command '['mineru', '-p', '../data/raw/Long-Tail Internet Photo Reconstruction.pdf', '-o', 'experiments/outputs/mineru/Long-Tail Internet Photo Reconstruction', '--backend', 'hybrid-auto-engine', '-f', 'True', '-t', 'True']' returned non-zero exit status 1.
Run IDs nested by paper: {'P1': {'mineru': 'fc3d0e783b0044589e75d1b1b9797fef'}, 'P2': {'mineru': '66c4f72f39d149708b8fa8ef468f176c'}}


,paper_id,file_name,parser,parser_version,run_id
0,P1,Relaxation-Informed Training of Neural Network...,mineru,3.1.5,fc3d0e783b0044589e75d1b1b9797fef
1,P2,Long-Tail Internet Photo Reconstruction.pdf,mineru,3.1.5,66c4f72f39d149708b8fa8ef468f176c


In [6]:
#parse_with_mineru(target_filepath)

### Update runs after manual inspection

In [7]:
# After manual review, log scores by paper_id + parser (no run ID copy-paste):

# log_manual_inspection_by_parser("pymupdf4llm", 2, "images omitted. sections and equations are not preserved", run_ids, paper_id="P1")
# log_manual_inspection_by_parser("docling", 4, "Good structure preservation. Equations partially preserved", run_ids, paper_id="P1")
# log_manual_inspection_by_parser("mineru", 5, "Best so far. Preserves images, tables and equations", run_ids, paper_id="P1")

# Example quick scoring loop (fill with your actual scores/comments):
# for paper_id in run_ids:
#     log_manual_inspection_by_parser("mineru", 5, "Strong overall for this paper", run_ids, paper_id=paper_id)

run_results_df

,paper_id,file_name,parser,parser_version,run_id
0,P1,Relaxation-Informed Training of Neural Network...,mineru,3.1.5,fc3d0e783b0044589e75d1b1b9797fef
1,P2,Long-Tail Internet Photo Reconstruction.pdf,mineru,3.1.5,66c4f72f39d149708b8fa8ef468f176c
